# ⚽ FC 26 Player Scouting & Valuation Analysis

An interactive Jupyter notebook for exploring **18,405 FC 26 players** using DuckDB, SQL, Plotly, KMeans and a linear regression valuation model.

The notebook is designed to be the **analysis + visualization layer** of the project, so there is no Streamlit app required. Charts are interactive (hover, zoom, pan, legend toggles), and the scouting section includes a searchable player selector with filters.

### What this notebook covers
- Market-value drivers with interactive correlation analysis
- Position economics and league wage analysis
- Career arc by age
- Player archetypes with KMeans clustering
- **Linear regression only** for market-value prediction
- Under/over-valued players from regression residuals
- Interactive player search, profile, attribute radar and valuation estimate

In [1]:
# Core libraries
import duckdb
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from IPython.display import display, HTML
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import db

# Optional notebook widgets. If this import fails in your environment, run:
# %pip install ipywidgets
try:
    import ipywidgets as widgets
    from IPython.display import clear_output
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
    print("ipywidgets is not installed. Run `%pip install ipywidgets`, restart the kernel, and rerun this cell.")

con = db.get_connection()
players = con.execute("SELECT * FROM players").df()

print(f"Loaded {len(players):,} players and {players.shape[1]} columns.")

Loaded 18,405 players and 110 columns.


## 1. Dataset overview

The FC 26 export contains detailed player, club, league, rating, contract and attribute fields. Goalkeepers do not have the six outfield attributes (`pace`, `shooting`, `passing`, `dribbling`, `defending`, `physic`), so outfield-only analyses filter them out rather than treating those values as zero.

In [2]:
summary = pd.DataFrame({
    "Metric": [
        "Players", "Columns", "Goalkeepers", "Outfield players",
        "Average overall", "Average potential"
    ],
    "Value": [
        f"{len(players):,}",
        f"{players.shape[1]}",
        f"{players['pace'].isna().sum():,}",
        f"{players['pace'].notna().sum():,}",
        f"{players['overall'].mean():.1f}",
        f"{players['potential'].mean():.1f}",
    ]
})
display(summary)

display(players[["short_name", "age", "overall", "potential", "value_eur", "wage_eur", "player_positions", "club_name", "league_name"]].head(10))

,Metric,Value
0,Players,"18,405"
1,Columns,110
2,Goalkeepers,"2,062"
3,Outfield players,"16,343"
4,Average overall,65.8
5,Average potential,71.2


,short_name,age,overall,potential,value_eur,wage_eur,player_positions,club_name,league_name
0,J. Bellingham,22,90,94,174500000,320000,"CAM, CM",Real Madrid,La Liga
1,F. Valverde,26,89,90,120500000,340000,"CM, CDM, RB",Real Madrid,La Liga
2,J. Kimmich,30,89,89,86000000,140000,"CDM, RB, CM",FC Bayern München,Bundesliga
3,A. Hakimi,26,89,90,111000000,170000,"RB, RM",Paris Saint-Germain,Ligue 1
4,N. Barella,28,87,87,79500000,69000,CM,Inter,Serie A
5,H. Çalhanoğlu,31,86,86,47500000,61000,"CDM, CM",Inter,Serie A
6,F. Dimarco,27,85,85,52000000,54000,"LB, LM",Inter,Serie A
7,A. Mac Allister,26,87,88,93000000,180000,"CM, CDM",Liverpool,Premier League
8,Rodri,29,90,90,102000000,270000,"CDM, CM",Manchester City,Premier League
9,A. Griezmann,34,85,85,26000000,115000,"ST, LM, CAM",Atlético Madrid,La Liga


## 2. Which attributes are associated with market value?

DuckDB calculates Pearson correlation directly in SQL. The chart below is Plotly-based, so you can hover over each bar, zoom, and toggle traces from the legend.

In [3]:
corr_df = db.run_query("attribute_value_correlations", con)
corr_long = corr_df.T.reset_index()
corr_long.columns = ["attribute", "correlation"]
corr_long["attribute"] = corr_long["attribute"].str.replace("_corr", "", regex=False)
corr_long = corr_long.sort_values("correlation")

fig = px.bar(
    corr_long,
    x="correlation",
    y="attribute",
    orientation="h",
    text="correlation",
    title="Correlation with market value",
    labels={"correlation": "Pearson correlation", "attribute": "Attribute"},
    hover_data={"correlation": ":.3f"},
)
fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.update_layout(height=520)
fig.show()

corr_long.sort_values("correlation", ascending=False)

,attribute,correlation
6,overall,0.554
7,potential,0.503
3,dribbling,0.411
2,passing,0.410
1,shooting,0.283
5,physic,0.225
0,pace,0.191
4,defending,0.159
8,age,0.026


## 3. Position economics

Players with multiple listed positions are counted in every position they can play. This uses `player_positions`, not `club_position`, so bench/reserve labels do not distort the analysis.

In [4]:
value_pos_df = db.run_query("value_by_position", con)
roi_df = db.run_query("roi_by_position", con)

fig_value = px.bar(
    value_pos_df.sort_values("avg_value_eur"),
    x="avg_value_eur", y="position", orientation="h",
    text="avg_value_eur",
    title="Average market value by position",
    labels={"avg_value_eur":"Average value (€)", "position":""},
    hover_data=["player_count", "avg_overall", "avg_wage_eur"],
)
fig_value.update_traces(texttemplate="€%{text:,.0f}", textposition="outside")
fig_value.update_layout(height=620)
fig_value.show()

fig_roi = px.bar(
    roi_df.sort_values("avg_value_per_overall_point"),
    x="avg_value_per_overall_point", y="position", orientation="h",
    text="avg_value_per_overall_point",
    title="Average value per overall-rating point",
    labels={"avg_value_per_overall_point":"€ per overall point", "position":""},
    hover_data=["player_count"],
)
fig_roi.update_traces(texttemplate="€%{text:,.0f}", textposition="outside")
fig_roi.update_layout(height=620)
fig_roi.show()

## 4. League wage economics

This view asks: **which leagues pay the most wage per overall-rating point?** Hover over a bar for player count, average overall and average wage.

In [5]:
league_df = db.run_query("league_pay_vs_performance", con)

fig = px.bar(
    league_df.sort_values("wage_per_overall_point"),
    x="wage_per_overall_point",
    y="league_name",
    orientation="h",
    text="wage_per_overall_point",
    title="Wage per overall-rating point by league",
    labels={"wage_per_overall_point":"€ wage per overall point", "league_name":""},
    hover_data=["player_count", "avg_overall", "avg_wage_eur"],
)
fig.update_traces(texttemplate="€%{text:,.0f}", textposition="outside")
fig.update_layout(height=720)
fig.show()

## 5. Career arc: rating, potential and value by age

In [6]:
age_df = db.run_query("age_curve", con)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=age_df["age"], y=age_df["avg_overall"], mode="lines+markers",
    name="Average overall", hovertemplate="Age %{x}<br>Overall %{y:.1f}<extra></extra>"
))
fig.add_trace(go.Scatter(
    x=age_df["age"], y=age_df["avg_potential"], mode="lines+markers",
    name="Average potential", hovertemplate="Age %{x}<br>Potential %{y:.1f}<extra></extra>"
))
fig.update_layout(
    title="Average overall vs. potential by age",
    xaxis_title="Age", yaxis_title="Rating",
    hovermode="x unified", height=520
)
fig.show()

fig_value_age = px.line(
    age_df, x="age", y="avg_value_eur", markers=True,
    title="Average market value by age",
    labels={"avg_value_eur":"Average value (€)", "age":"Age"},
    hover_data=["player_count", "avg_overall", "avg_potential"],
)
fig_value_age.update_yaxes(tickformat="~s")
fig_value_age.show()

## 6. Player archetypes with KMeans

KMeans uses the six core outfield attributes. Standardization is applied first so the clustering is based on relative player profiles rather than the scale of a feature.

In [7]:
CLUSTER_FEATURES = ["pace", "shooting", "passing", "dribbling", "defending", "physic"]
outfield = players.dropna(subset=CLUSTER_FEATURES).copy()
X_scaled = StandardScaler().fit_transform(outfield[CLUSTER_FEATURES])

# Elbow curve
inertias = []
ks = list(range(2, 9))
for k in ks:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertias.append(model.inertia_)

fig = px.line(
    x=ks, y=inertias, markers=True,
    title="KMeans elbow curve",
    labels={"x":"Number of clusters (k)", "y":"Inertia"}
)
fig.show()

# Default model
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
outfield["cluster"] = kmeans.fit_predict(X_scaled).astype(str)
cluster_summary = outfield.groupby("cluster")[CLUSTER_FEATURES + ["overall", "value_eur"]].mean().round(1).reset_index()
display(cluster_summary)

fig = px.scatter(
    outfield.sample(min(8000, len(outfield)), random_state=42),
    x="defending", y="shooting",
    color="cluster",
    hover_data=["short_name", "player_positions", "overall", "potential", "value_eur"],
    title="Player archetypes: defending vs. shooting",
    opacity=0.55,
)
fig.update_layout(height=650)
fig.show()

,cluster,pace,shooting,passing,dribbling,defending,physic,overall,value_eur
0,0,68.5,53.9,52.2,60.2,36.2,54.6,59.7,633745.7
1,1,68.9,56.0,64.6,67.6,64.7,70.1,69.7,4519742.4
2,2,59.7,34.1,47.4,51.1,61.8,67.9,62.9,1105840.7
3,3,77.2,67.5,63.7,71.4,37.5,64.2,70.5,5717033.7


## 7. Market-value prediction — linear regression

The target is `log(value_eur)` because player values are heavily right-skewed. The model is evaluated on a held-out 20% test set so the reported metrics measure out-of-sample performance.

**Only linear regression is used in this notebook.** The goal is an interpretable baseline that can also be used to estimate a player's expected value and identify large residuals for scouting.

In [8]:
VALUE_FEATURES = [
    "overall", "potential", "age",
    "pace", "shooting", "passing", "dribbling", "defending", "physic"
]

reg_df = players.dropna(subset=VALUE_FEATURES + ["value_eur"]).copy()
reg_df = reg_df[reg_df["value_eur"] > 0].copy()

X = reg_df[VALUE_FEATURES]
y = np.log(reg_df["value_eur"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

lr = LinearRegression()
lr.fit(X_train, y_train)

pred_test_log = lr.predict(X_test)
pred_test_eur = np.exp(pred_test_log)
actual_test_eur = np.exp(y_test)

metrics = {
    "R²": r2_score(y_test, pred_test_log),
    "MAE (€)": mean_absolute_error(actual_test_eur, pred_test_eur),
}
print(f"Test R²: {metrics['R²']:.3f}")
print(f"Test MAE: €{metrics['MAE (€)']:,.0f}")

coef_df = pd.DataFrame({
    "feature": VALUE_FEATURES,
    "coefficient": lr.coef_
}).sort_values("coefficient")

fig = px.bar(
    coef_df, x="coefficient", y="feature", orientation="h",
    text="coefficient",
    title="Linear regression coefficients for log(market value)",
    labels={"coefficient":"Coefficient", "feature":"Feature"},
)
fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.update_layout(height=560)
fig.show()

Test R²: 0.971
Test MAE: €679,717


In [9]:
# Actual vs predicted values on the held-out test set
pred_plot = pd.DataFrame({
    "actual_value_eur": actual_test_eur.values,
    "predicted_value_eur": pred_test_eur,
    "player_index": X_test.index,
}).merge(
    players[["short_name", "club_name", "player_positions", "overall", "potential"]],
    left_on="player_index", right_index=True, how="left"
)

fig = px.scatter(
    pred_plot,
    x="actual_value_eur",
    y="predicted_value_eur",
    hover_data=["short_name", "club_name", "player_positions", "overall", "potential"],
    log_x=True, log_y=True,
    title="Actual vs. predicted market value — held-out test set",
    labels={"actual_value_eur":"Actual value (€)", "predicted_value_eur":"Predicted value (€)"},
)
fig.add_shape(
    type="line", x0=pred_plot["actual_value_eur"].min(),
    y0=pred_plot["actual_value_eur"].min(),
    x1=pred_plot["actual_value_eur"].max(),
    y1=pred_plot["actual_value_eur"].max(),
    line=dict(dash="dash")
)
fig.update_layout(height=620)
fig.show()

## 8. Under-valued and over-valued scouting candidates

For each player, the residual is:

`actual log(value) - predicted log(value)`

A strongly **negative** residual means the player is valued below the model's estimate; a strongly **positive** residual means the actual value is above the model's estimate.

In [10]:
reg_df["predicted_log_value"] = lr.predict(reg_df[VALUE_FEATURES])
reg_df["predicted_value_eur"] = np.exp(reg_df["predicted_log_value"])
reg_df["residual"] = np.log(reg_df["value_eur"]) - reg_df["predicted_log_value"]
reg_df["value_gap_eur"] = reg_df["predicted_value_eur"] - reg_df["value_eur"]

undervalued = reg_df.sort_values("residual").head(15)
overvalued = reg_df.sort_values("residual", ascending=False).head(15)

cols = ["short_name", "age", "club_name", "league_name", "player_positions", "overall", "potential", "value_eur", "predicted_value_eur", "value_gap_eur", "residual"]

display(undervalued[cols].style.format({
    "value_eur":"€{:,.0f}", "predicted_value_eur":"€{:,.0f}", "value_gap_eur":"€{:,.0f}", "residual":"{:.3f}"
}))

fig = px.scatter(
    reg_df.sample(min(8000, len(reg_df)), random_state=42),
    x="predicted_value_eur", y="value_eur",
    color="residual",
    hover_data=["short_name", "club_name", "player_positions", "overall", "potential", "age"],
    log_x=True, log_y=True,
    color_continuous_scale="RdBu",
    title="Predicted vs. actual value — residuals reveal scouting gaps",
    labels={"predicted_value_eur":"Predicted value (€)", "value_eur":"Actual value (€)", "residual":"Residual"},
)
fig.update_layout(height=700)
fig.show()

,short_name,age,club_name,league_name,player_positions,overall,potential,value_eur,predicted_value_eur,value_gap_eur,residual
3058,A. Corzo,36,Universitario de Deportes,Liga 1,"CB, RB",71,71,"€325,000","€915,331","€590,331",-1.035
8048,E. Aguilera,36,Defensa y Justicia,Liga Profesional de Fútbol,CB,71,71,"€325,000","€901,884","€576,884",-1.021
3931,S. Vittor,36,CA Banfield,Liga Profesional de Fútbol,CB,70,70,"€275,000","€751,035","€476,035",-1.005
11027,Aridane,36,UD Almería,La Liga 2,CB,71,71,"€325,000","€856,236","€531,236",-0.969
7211,M. Boxall,36,Minnesota United FC,Major League Soccer,CB,70,70,"€275,000","€722,956","€447,956",-0.967
8118,M. Yoshida,36,LA Galaxy,Major League Soccer,CB,70,70,"€275,000","€722,825","€447,825",-0.966
12409,M. Danielson,36,Djurgårdens IF,Allsvenskan,CB,70,70,"€275,000","€720,223","€445,223",-0.963
6983,Aderllan Santos,36,AVS Futebol SAD,Primeira Liga,CB,72,72,"€425,000","€1,093,017","€668,017",-0.945
1265,Petros,36,Al Akhdoud Saudi Club,Pro League,"CDM, CM",72,72,"€450,000","€1,131,807","€681,807",-0.922
8464,C. Henao,36,Atlético Bucaramanga,Categoría Primera A,CB,69,69,"€250,000","€615,738","€365,738",-0.901


## 9. 🔎 Interactive player search & scouting profile

Use the search bar below to select a player. The widget combines:
- player search/autocomplete
- key rating and valuation metrics
- actual vs. regression-predicted value
- attribute radar chart
- closest players by standardized core attributes

**Notebook requirement:** `ipywidgets` must be enabled in your Jupyter environment. If the widget does not render, run `%pip install ipywidgets` and restart the kernel.

In [11]:
if not WIDGETS_AVAILABLE:
    display(HTML("""
    <div style='padding:12px;border:1px solid #ddd;border-radius:8px'>
    <b>Interactive player search is unavailable.</b><br>
    Install ipywidgets with <code>%pip install ipywidgets</code>, restart the kernel, and rerun this section.
    </div>
    """))
else:
    player_names = sorted(players["short_name"].dropna().unique().tolist())
    player_box = widgets.Combobox(
        options=player_names,
        placeholder="Type a player name…",
        description="Player:",
        ensure_option=False,
        layout=widgets.Layout(width="520px"),
    )
    position_options = ["All"] + sorted({p.strip() for s in players["player_positions"].dropna() for p in s.split(",") if p.strip()})
    position_box = widgets.Dropdown(options=position_options, description="Position:", layout=widgets.Layout(width="280px"))
    min_pot = widgets.IntSlider(value=0, min=0, max=99, step=1, description="Min POT:", continuous_update=False, layout=widgets.Layout(width="320px"))
    max_value = widgets.IntSlider(value=200, min=5, max=200, step=5, description="Max €M:", continuous_update=False, layout=widgets.Layout(width="320px"))

    out = widgets.Output()
    sim_features = ["pace", "shooting", "passing", "dribbling", "defending", "physic"]
    sim_df = players.dropna(subset=sim_features).copy()
    sim_scaled = StandardScaler().fit_transform(sim_df[sim_features])

    def format_eur(x):
        if pd.isna(x): return "—"
        if x >= 1_000_000_000: return f"€{x/1_000_000_000:.2f}B"
        if x >= 1_000_000: return f"€{x/1_000_000:.1f}M"
        if x >= 1_000: return f"€{x/1_000:.0f}K"
        return f"€{x:,.0f}"

    def show_player(*_):
        with out:
            clear_output(wait=True)
            name = player_box.value.strip()
            if not name:
                print("Start typing a player name above.")
                return

            matches = players[players["short_name"].str.lower().str.contains(name.lower(), na=False)]
            if position_box.value != "All":
                matches = matches[matches["player_positions"].fillna("").str.contains(position_box.value, regex=False)]
            matches = matches[matches["potential"] >= min_pot.value]
            matches = matches[matches["value_eur"] <= max_value.value * 1_000_000]

            if matches.empty:
                print("No player matches those filters. Try a broader search.")
                return

            # Prefer exact name match, otherwise first result
            exact = matches[matches["short_name"].str.lower() == name.lower()]
            player = (exact.iloc[0] if not exact.empty else matches.iloc[0])

            row = reg_df[reg_df.index == player.name]
            if row.empty:
                predicted = np.nan
            else:
                predicted = float(row["predicted_value_eur"].iloc[0])

            growth = player["potential"] - player["overall"]
            display(HTML(f"""
            <div style='padding:14px;border:1px solid #ddd;border-radius:10px;margin:10px 0'>
              <h3 style='margin:0 0 10px 0'>⚽ {player['short_name']}</h3>
              <b>{player['club_name']}</b> · {player['league_name']} · {player['player_positions']}<br>
              Age <b>{int(player['age'])}</b> · OVR <b>{int(player['overall'])}</b> · POT <b>{int(player['potential'])}</b> · Growth <b>+{int(growth)}</b><br>
              Market value <b>{format_eur(player['value_eur'])}</b> · Wage <b>{format_eur(player['wage_eur'])}</b> · Predicted value <b>{format_eur(predicted)}</b>
            </div>
            """))

            # Attribute radar
            radar_values = [float(player[f]) for f in sim_features]
            radar_labels = ["Pace", "Shooting", "Passing", "Dribbling", "Defending", "Physical"]
            fig_radar = go.Figure()
            fig_radar.add_trace(go.Scatterpolar(
                r=radar_values + [radar_values[0]],
                theta=radar_labels + [radar_labels[0]],
                fill="toself", name=player["short_name"]
            ))
            fig_radar.update_layout(
                title=f"{player['short_name']} — attribute profile",
                polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
                height=480,
            )
            fig_radar.show()

            # Similar players using standardized Euclidean distance
            pvec = StandardScaler().fit(sim_df[sim_features]).transform(sim_df[sim_features][sim_df.index == player.name]) if player.name in sim_df.index else None
            if pvec is not None and len(pvec):
                # Reuse the already-scaled matrix created above
                distances = np.linalg.norm(sim_scaled - pvec[0], axis=1)
                sim_out = sim_df.copy()
                sim_out["distance"] = distances
                similar = sim_out[sim_out.index != player.name].sort_values("distance").head(8)
                similar = similar[["short_name", "club_name", "player_positions", "overall", "potential", "value_eur", "distance"]]
                display(similar.style.format({"value_eur":"€{:,.0f}", "distance":"{:.2f}"}))

    player_box.observe(show_player, names="value")
    position_box.observe(show_player, names="value")
    min_pot.observe(show_player, names="value")
    max_value.observe(show_player, names="value")

    display(widgets.VBox([
        widgets.HBox([player_box]),
        widgets.HBox([position_box, min_pot, max_value]),
        out
    ]))
    print("Tip: start typing a name such as Bellingham, Mbappé, or Yamal.")

Tip: start typing a name such as Bellingham, Mbappé, or Yamal.


## 10. Interactive scouting table

This final view applies the same filters to the full player pool and gives a compact table for recruitment shortlists.

In [ ]:
if not WIDGETS_AVAILABLE:
    display(HTML("Interactive scouting filters require <code>ipywidgets</code>."))
else:
    pos_filter = widgets.Dropdown(options=position_options, value="All", description="Position:")
    pot_filter = widgets.IntSlider(value=75, min=0, max=99, description="Min POT:", continuous_update=False)
    age_filter = widgets.IntRangeSlider(value=(16, 35), min=16, max=40, description="Age:", continuous_update=False)
    value_filter = widgets.IntSlider(value=50, min=1, max=200, step=1, description="Max €M:", continuous_update=False)
    shortlist_out = widgets.Output()

    def refresh_shortlist(*_):
        with shortlist_out:
            clear_output(wait=True)
            q = players.copy()
            if pos_filter.value != "All":
                q = q[q["player_positions"].fillna("").str.contains(pos_filter.value, regex=False)]
            q = q[q["potential"] >= pot_filter.value]
            q = q[q["age"].between(age_filter.value[0], age_filter.value[1])]
            q = q[q["value_eur"] <= value_filter.value * 1_000_000]
            q = q.sort_values(["potential", "value_eur"], ascending=[False, True]).head(50)
            table_cols = ["short_name", "age", "player_positions", "overall", "potential", "value_eur", "wage_eur", "club_name", "league_name"]
            display(q[table_cols].style.format({"value_eur":"€{:,.0f}", "wage_eur":"€{:,.0f}"}))
            print(f"{len(q)} players shown (top 50 by potential, then lowest value).")

    for w in [pos_filter, pot_filter, age_filter, value_filter]:
        w.observe(refresh_shortlist, names="value")

    display(widgets.VBox([
        widgets.HBox([pos_filter, pot_filter]),
        widgets.HBox([age_filter, value_filter]),
        shortlist_out
    ]))
    refresh_shortlist()

: 

## Final takeaways

The project now lives entirely in the notebook: SQL handles aggregation and filtering, scikit-learn handles KMeans and the single linear regression model, Plotly handles interactive visualization, and ipywidgets handles player search and scouting filters.

### Model caveat
The regression is a **scouting signal, not a transfer-price truth**. The model uses player attributes, age, overall and potential; it does not know the full context behind a real-world transfer valuation such as contract situation, club finances, reputation or transfer-market dynamics.